# 🚀 Training ACD-MMKTv3 (Full Model & Ablation Study Suite) on Eedi — Kaggle Dual T4 GPUs

**Mô hình:** Asymmetric Causal-Disentangled Multimodal Knowledge Tracing v3 (**ACD-MMKTv3**)  
**Tập dữ liệu:** Preprocessed **Eedi NeurIPS 2020 (Task 3 & 4)** — 1,508,917 tương tác, 6,148 học sinh (4,918 Train/Val trong 5 Folds + 1,230 Test độc lập: 615 Public + 615 Private = **124,960** bước đánh giá hợp nhất `unified`), 948 câu hỏi chẩn đoán toán học & 85 KCs.  
**Đặc trưng Eedi trong ACD-MMKTv3:**  
- **Mặt nạ phương thức thực tế:** Ảnh đề bài `m_img^q = 495/948` (sau lọc trùng/huy hiệu trên 948 slide), Văn bản lời giải tổng hợp VLM `m_txt^s = 948/948`, Ảnh lời giải `m_img^s = 0/948` (tự động tắt nhánh $I_s$).  
- **Vector chẩn đoán hậu phản hồi (`post_diag_mode = "diagnostic"`):** 8 chiều nhân quả từ đáp án lựa chọn A/B/C/D, cờ chọn nhầm misconception chủ đạo, mức tự tin và lỗi tự tin quá mức.  
- **Đánh giá đặc thù Eedi:** Tích hợp kiểm định 50 cặp chất lượng câu hỏi của chuyên gia **Task 3**, phân tầng 3 hình thái slide (`Horizontal`, `Grid 2x2`, `Vertical`), 3 miền toán học (`Number`, `Algebra`, `Geometry & Measure`), và hỗ trợ đánh giá đồng thời trên cả 3 tập Test: `unified` (124,960), `public` (74,276) và `private` (50,684).  
- **Tối ưu hóa phần cứng:** Hỗ trợ chạy song song bất đối xứng **2x NVIDIA Tesla T4 GPUs** (`--dual-gpu`, GPU 0 chạy Folds `[0, 1, 2]`, GPU 1 chạy Folds `[3, 4]`) với `amp_fp16` và tổng hợp **5-Fold Soft Probability Ensemble**.

---

### ⚙️ Hướng Dẫn Cấu Hình Kaggle Notebook (Bắt Buộc):
1. **Accelerator:** Trong bảng **Session options** bên phải, chọn **GPU T4 x 2** (2x NVIDIA Tesla T4, 16GB VRAM mỗi GPU).
2. **Internet:** Bật **Internet ON** để tải thư viện bổ sung và clone/pull repository.
3. **Data Input:** Nhấn **+ Add Input**, gắn Dataset **Eedi đã tiền xử lý** (từ notebook Colab `colab_preprocess_eedi_task34.ipynb`, chứa `static_item_bank.pt`, `acd_v3_aux_metadata.pt`, `sequences_fold_0..4.pt`, `sequences_test*.pt`, `test_strata_masks.pt`).
   *(Tùy chọn: Nếu bạn đã train xong một phần ở session trước và lưu thành Kaggle Dataset checkpoint, hãy Add Input luôn để notebook tự symlink và bỏ qua các fold đã hoàn tất).*
4. **Chỉnh tham số & chế độ chạy:** Toàn bộ cấu hình chế độ chạy (`EXECUTION_MODE`), siêu tham số (`NUM_EPOCHS`, `BATCH_SIZE`, `LR`, `POST_DIAG_MODE`, `TEST_SPLIT`) nằm tập trung tại **Cell [6]**.

In [ ]:
# [1] Kiểm tra cấu hình phần cứng & Thiết lập môi trường Dual GPU T4 / CUDA
import os
import sys
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

print(f"🔥 PyTorch Version: {torch.__version__}")

is_gpu = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if is_gpu else 0

if is_gpu and gpu_count >= 2:
    print(f"⚡ [ACCELERATOR: DUAL GPU T4] Đã nhận diện {gpu_count} GPUs:")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"   - [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | CC: {props.major}.{props.minor}")
    print("🚀 Sẵn sàng kích hoạt --dual-gpu: Worker 0 (cuda:0 -> Folds [0,1,2]) & Worker 1 (cuda:1 -> Folds [3,4])!")
    !nvidia-smi
elif is_gpu and gpu_count == 1:
    props = torch.cuda.get_device_properties(0)
    print(f"⚡ [ACCELERATOR: SINGLE GPU] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
    !nvidia-smi
else:
    print("⚠️ CẢNH BÁO: Không phát hiện GPU CUDA! Hãy chọn 'GPU T4 x 2' trong Session Options.")

assert is_gpu, "⚠️ Vui lòng bật GPU (khuyên dùng GPU T4 x 2) trong Kaggle Session Options trước khi chạy!"

In [ ]:
# [2] Thiết lập thư mục làm việc & Clone/Pull repository từ GitHub
import os
import sys

REPO_URL = "https://github.com/nbao0208/kt-mmkt-research.git"
REPO_DIR = "/kaggle/working/kt-mmkt-research"

if os.path.exists("src/models/mmkt/acd_mmkt_v3.py") and os.path.exists("src/data/dataset_router.py"):
    print("✅ Đang đứng trực tiếp trong thư mục repository kt-mmkt-research.")
    !git pull origin main
elif os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Đã chuyển đến: {REPO_DIR}")
    !git pull origin main
else:
    print(f"📥 Đang clone repository từ {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"✅ Đã clone thành công vào: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())

In [ ]:
# [3] Cài đặt các thư viện phụ thuộc (Dependencies)
!pip install -q -r requirements.txt
!pip install -q seaborn matplotlib scikit-learn tabulate pandas transformers scipy pyyaml
print("✅ Hoàn tất cài đặt toàn bộ gói phụ thuộc cho ACD-MMKTv3 (Eedi)!")

In [ ]:
# [4] Tự động phát hiện, liên kết (Symlink) và kiểm toán dữ liệu Eedi đã tiền xử lý từ /kaggle/input/
import glob
import os
from pathlib import Path
import torch

# 1. Ưu tiên tìm chữ ký đặc trưng của Eedi Task 3 & 4 (sequences_test_public.pt)
candidates = glob.glob("/kaggle/input/**/sequences_test_public.pt", recursive=True)
if not candidates:
    all_banks = glob.glob("/kaggle/input/**/static_item_bank.pt", recursive=True)
    eedi_banks = [p for p in all_banks if "eedi" in p.lower()]
    candidates = eedi_banks or all_banks

if not candidates:
    raise FileNotFoundError(
        "❌ Không tìm thấy dữ liệu Eedi đã tiền xử lý trong /kaggle/input/!\n"
        "👉 Vui lòng nhấn '+ Add Input' và gắn Dataset Eedi (chứa static_item_bank.pt, acd_v3_aux_metadata.pt, sequences_fold_*.pt, sequences_test*.pt)."
    )

input_data_dir = Path(candidates[0]).parent
print(f"🔍 Đã phát hiện thư mục dữ liệu Eedi tại: {input_data_dir}")

target_dir = Path("data/processed/Eedi")
target_dir.mkdir(parents=True, exist_ok=True)

# 2. Tạo symlink cho toàn bộ artifact (.pt, .json, .npy, .parquet, .csv) vào data/processed/Eedi/
linked_count = 0
for ext in ["*.pt", "*.json", "*.npy", "*.parquet", "*.csv"]:
    for src_file in input_data_dir.glob(ext):
        dst_file = target_dir / src_file.name
        if dst_file.exists() or dst_file.is_symlink():
            dst_file.unlink()
        dst_file.symlink_to(src_file)
        linked_count += 1

# Kiểm tra nếu người dùng upload cả thư mục cha chứa features/ và splits/
for sub_name, dst_sub in [("features", "data/features/Eedi"), ("splits", "data/splits/Eedi")]:
    for cand in glob.glob(f"/kaggle/input/**/{sub_name}/**/*", recursive=True):
        src_p = Path(cand)
        if src_p.is_file():
            dst_dir = Path(dst_sub)
            dst_dir.mkdir(parents=True, exist_ok=True)
            dst_p = dst_dir / src_p.name
            if dst_p.exists() or dst_p.is_symlink():
                dst_p.unlink()
            dst_p.symlink_to(src_p)

print(f"✅ Đã symlink {linked_count} tệp vào {target_dir}")

# 3. Kiểm tra toàn vẹn toàn bộ tệp bắt buộc cho ACD-MMKTv3 & Ablation Study trên Eedi
required_files = [
    "static_item_bank.pt",
    "acd_v3_aux_metadata.pt",
    "test_strata_masks.pt",
    "sequences_test.pt",
    "sequences_test_public.pt",
    "sequences_test_private.pt",
] + [f"sequences_fold_{f}.pt" for f in range(5)]

missing = [fn for fn in required_files if not (target_dir / fn).exists()]
if missing:
    raise FileNotFoundError(
        f"❌ Thiếu các tệp Eedi bắt buộc trong {target_dir}: {missing}\n"
        "👉 Vui lòng kiểm tra lại output của notebook tiền xử lý Colab (colab_preprocess_eedi_task34.ipynb)."
    )

# In tóm tắt xác thực kích thước tensor Eedi
bank = torch.load(target_dir / "static_item_bank.pt", map_location="cpu", weights_only=False)
aux = torch.load(target_dir / "acd_v3_aux_metadata.pt", map_location="cpu", weights_only=False)
masks = bank["modality_masks"]
print(f"📊 Xác nhận Eedi Item Bank:")
print(f"   - Số câu hỏi: {masks.shape[0]} | Số KCs: {bank['acd_hypergraph_G'].shape[0]}")
print(f"   - m_img^q (có ảnh đề): {int(masks[:, 0].sum())}/{masks.shape[0]} | m_txt^s (có lời giải text): {int(masks[:, 1].sum())}/{masks.shape[0]} | m_img^s (ảnh lời giải): {int(masks[:, 2].sum())}/{masks.shape[0]}")
print(f"   - Task 3 Expert Pairs: {len(aux.get('task3_expert_pairs', []))} cặp")

# 4. (Tùy chọn) Tự động symlink checkpoint Eedi có sẵn từ /kaggle/input/ nếu người dùng đính kèm để resume/eval
eedi_ckpt_root = Path("outputs/checkpoints/eedi")
eedi_ckpt_root.mkdir(parents=True, exist_ok=True)
variant_folders = [
    "acd_mmkt_v3_full_5fold",
    "acd_mmkt_v3_ablation3_wo_visual_5fold",
    "acd_mmkt_v3_ablation2_wo_solution_5fold",
    "acd_mmkt_v3_ablation1_wo_all_multimodal_5fold",
    "acd_mmkt_v3_ablation4_unanchored_5fold",
    "acd_mmkt_v3_ablation5_polluted_exp_5fold",
    "acd_mmkt_v3_ablation6_wo_2pl_irt_5fold",
]
restored = 0
for v_name in variant_folders:
    found = [p for p in glob.glob(f"/kaggle/input/**/{v_name}", recursive=True)]
    if found:
        src_v = Path(found[0])
        dst_v = eedi_ckpt_root / v_name
        for f in range(5):
            f_src = src_v / f"acd_mmkt_v3_fold{f}"
            if f_src.exists():
                f_dst = dst_v / f"acd_mmkt_v3_fold{f}"
                f_dst.mkdir(parents=True, exist_ok=True)
                for item in f_src.glob("*"):
                    d_item = f_dst / item.name
                    if d_item.exists() or d_item.is_symlink():
                        d_item.unlink()
                    d_item.symlink_to(item)
                    restored += 1
if restored > 0:
    print(f"📦 Đã phát hiện và liên kết {restored} tệp checkpoint Eedi có sẵn vào {eedi_ckpt_root}")

In [ ]:
# [5] Chạy Sanity Check Unit & Integration Tests trên dữ liệu Eedi thật
import subprocess
import sys

print("🧪 Đang chạy bộ kiểm thử Eedi Pipeline, Preprocessing Gates, Ablation Purity & pyKT Protocol...")
cmd = [
    sys.executable, "-m", "pytest",
    "tests/test_eedi_acd_mmkt_v3_pipeline.py",
    "tests/test_eedi_task34_preprocessing.py",
    "tests/test_acd_mmkt_v3_ablation_study.py",
    "tests/test_pykt_protocol.py",
    "-v",
]
res = subprocess.run(cmd)
if res.returncode == 0:
    print("\n🎉 TẤT CẢ CÁC BÀI KIỂM THỬ ĐÃ PASSED 100%! Hệ thống sẵn sàng huấn luyện trên Dual GPU T4.")
else:
    raise RuntimeError(f"❌ Kiểm thử thất bại với mã lỗi {res.returncode}. Vui lòng kiểm tra log phía trên.")

## 🔬 Đặc Tả Ma Trận Thực Nghiệm ACD-MMKTv3 Trên Eedi (`d_model=128`, `post_diag_mode="diagnostic"`)

Tất cả biến thể cắt bỏ đều tuân thủ tuyệt đối nguyên tắc **Pure Pruning** & **Zero Modality Leakage** (không cấy tham số ngoại lai, giữ nguyên vector lịch sử chẩn đoán hậu phản hồi 8 chiều `post_diag_mode="diagnostic"` để đảm bảo đối chứng công bằng *ceteris paribus*):

| Biến thể / Cấu hình | Cờ `--ablation` | Phương thức giữ lại | Số tham số khả vi (Eedi) | Chênh lệch so với Full |
| :--- | :---: | :--- | :---: | :---: |
| **Full ACD-MMKTv3** | `full` / `0` | $C_\tau, T_q, I_q, T_s$ *(Eedi không có $I_s$)* + `post_diag` | **762,376** | Mốc đối chiếu (0%) |
| **Ablation 2 (`w/o Solution`)** | `2` | $C_\tau, T_q, I_q$ (bỏ lời giải $T_s$, giữ ảnh đề $I_q$) | **729,352** | **-4.33%** (-33,024) |
| **Ablation 3 (`w/o Visual`)** | `3` | $C_\tau, T_q, T_s$ (bỏ hoàn toàn nhánh ảnh $I_q$, $\bar{\lambda}_q = 0$) | **530,056** | **-30.47%** (-232,320) |
| **Ablation 1 (`Pure KC`)** | `1` | Chỉ $C_\tau$ + Siêu đồ thị $\mathbf{G}$ ($d^q = \mathbf{0}, \beta_d = 0, \lambda_{\text{sparse}} = 0$) | **211,592** | **-72.25%** (-550,784) |

*(Tùy chọn nâng cao khi chọn `EXECUTION_MODE = "ALL_ABLATIONS_FULL_SUITE"`: có thể chạy thêm 3 biến thể cắt bỏ kiến trúc — Ablation 4 `Unanchored Concept` [762,505], Ablation 5 `Polluted Experience` [828,680], Ablation 6 `w/o 2PL-IRT` [745,607]).*

### ⚡ Chiến Lược Lập Lịch Song Song Dual NVIDIA Tesla T4:
- Cờ `--dual-gpu` chia tự động 5-Fold CV thành 2 tiến trình song song không đồng bộ:
  - **Worker 0 (`cuda:0`):** Folds **[0, 1, 2]**
  - **Worker 1 (`cuda:1`):** Folds **[3, 4]**
- Ngay sau khi cả 2 Worker hoàn tất, hệ thống tự động tính **5-Fold Soft Probability Ensemble** và lưu toàn bộ kết quả cô lập hoàn toàn trong `outputs/checkpoints/eedi/` và `outputs/reports/eedi/`.

In [ ]:
# ==============================================================================
# [6] CẤU HÌNH HUẤN LUYỆN TOÀN CỤC CHO EEDI (PARAMETER OVERRIDES)
# ==============================================================================
import os

# --- 1. Chế độ thực thi (Execution Mode) ---
# "FULL_AND_MODALITY_ABLATIONS": Huấn luyện Full + 3 Modality Ablations (3, 2, 1) (Khuyên dùng)
# "FULL_ONLY"                  : Chỉ huấn luyện bản Full ACD-MMKTv3 (5-Fold CV)
# "MODALITY_ABLATIONS_ONLY"    : Chỉ huấn luyện 3 Modality Ablations (3, 2, 1) (cần có Full từ trước)
# "ALL_ABLATIONS_FULL_SUITE"   : Full + Modality Ablations (3, 2, 1) + Architecture Ablations (4, 5, 6)
# "EVAL_AND_SYNTHESIS_ONLY"    : Bỏ qua huấn luyện, chỉ chạy đánh giá + tổng hợp trên checkpoint sẵn có
# "SINGLE_FOLD"                : Chỉ huấn luyện 1 fold (TARGET_FOLD) cho bản Full
# "QUICK_TEST"                 : Smoke test 1 epoch trên Fold 0 để kiểm tra nhanh luồng chạy
EXECUTION_MODE = "FULL_AND_MODALITY_ABLATIONS"
TARGET_FOLD = 0
FORCE_RETRAIN = False    # Đặt True nếu muốn train đè dù đã thấy best_model.pt của tất cả 5 fold

# --- 2. CẤU HÌNH ĐẶC THÙ EEDI ---
DATASET = "eedi"
POST_DIAG_MODE = "diagnostic"   # 'diagnostic' (8 chiều A-D/misconception/confidence), 'behavioral', hoặc 'none'
TEST_SPLIT = "unified"          # Tập test chính: 'unified' (124,960), 'public' (74,276), 'private' (50,684)
EVAL_PUBLIC_PRIVATE = True      # Tự động chấm thêm báo cáo riêng cho 'public' và 'private' sau khi train xong

# --- 3. THAM SỐ HUẤN LUYỆN (ĐỒNG BỘ CETERIS PARIBUS VỚI XES3G5M) ---
NUM_EPOCHS = 50
BATCH_SIZE = 32                 # Batch size trên mỗi GPU T4 (VRAM 16GB)
LR = 1e-3
GRAD_ACCUM_STEPS = 4            # Effective Batch Size = 32 * 4 = 128
PRECISION = "amp_fp16"          # Tối ưu cho NVIDIA Tesla T4
NUM_WORKERS = 2
PATIENCE = 10
PROFILE = "kaggle_t4"           # Profile trong configs/train_acd_mmkt_v3_eedi.yaml

# --- 4. THAM SỐ ĐÁNH GIÁ ---
TEST_BATCH_SIZE = 64
TEST_STRIDE = 100

# --- 5. THƯ MỤC ĐẦU RA CÔ LẬP CHO EEDI ---
CKPT_BASE = "outputs/checkpoints/eedi"
REPORT_BASE = "outputs/reports/eedi"
LOG_BASE = "outputs/logs/eedi"
FIG_BASE = "outputs/reports/eedi/figures"
for d in [CKPT_BASE, REPORT_BASE, LOG_BASE, FIG_BASE]:
    os.makedirs(d, exist_ok=True)

# --- 6. TỰ ĐỘNG THIẾT LẬP CỜ DUAL-GPU T4 ---
if is_gpu and gpu_count >= 2:
    ACCEL_FLAG = "--dual-gpu"
    DEVICE_STR = "cuda:0"
    print("🚀 [DUAL-GPU T4]: Kích hoạt '--dual-gpu' (GPU 0: Folds [0,1,2] | GPU 1: Folds [3,4])")
else:
    ACCEL_FLAG = ""
    DEVICE_STR = "cuda:0" if is_gpu else "cpu"
    print(f"ℹ️ [SINGLE-DEVICE]: Chạy tuần tự trên {DEVICE_STR}")

def all_5folds_exist(exp_folder: str) -> bool:
    return all(
        os.path.exists(os.path.join(CKPT_BASE, exp_folder, f"acd_mmkt_v3_fold{f}", "best_model.pt"))
        for f in range(5)
    )

print("=" * 74)
print(f"📋 CẤU HÌNH ĐANG ÁP DỤNG (DATASET={DATASET.upper()}):")
print(f"   - Mode: {EXECUTION_MODE} | Profile: {PROFILE} | Post-Diag: {POST_DIAG_MODE}")
print(f"   - Test Split: {TEST_SPLIT} | Eval Public+Private: {EVAL_PUBLIC_PRIVATE}")
print(f"   - Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} x Accum {GRAD_ACCUM_STEPS} (Eff={BATCH_SIZE * GRAD_ACCUM_STEPS}) | LR: {LR}")
print(f"   - Precision: {PRECISION} | Workers: {NUM_WORKERS} | Patience: {PATIENCE}")
print(f"   - Outputs: {CKPT_BASE} | {REPORT_BASE}")
print("=" * 74)

In [ ]:
# ==============================================================================
# [7] Huấn Luyện Full ACD-MMKTv3 trên Eedi (5-Fold CV Dual-GPU / Single / Smoke)
# ==============================================================================
import os

RUN_5FOLD_MODES = {"FULL_AND_MODALITY_ABLATIONS", "FULL_ONLY", "ALL_ABLATIONS_FULL_SUITE"}

if EXECUTION_MODE in RUN_5FOLD_MODES:
    eval_flag = "--eval-only" if (not FORCE_RETRAIN and all_5folds_exist("acd_mmkt_v3_full_5fold")) else ""
    if eval_flag:
        print("📦 Đã có đủ 5 checkpoints Full ACD-MMKTv3 trên Eedi -> chạy --eval-only để tạo predictions & report.")
    else:
        print("🚀 Bắt đầu huấn luyện Full ACD-MMKTv3 (5-Fold CV) trên Eedi...")

    !python -u scripts/run_5fold_acd_mmkt_v3.py \
        --dataset {DATASET} \
        --test-split {TEST_SPLIT} \
        --post-diag-mode {POST_DIAG_MODE} \
        {ACCEL_FLAG} \
        {eval_flag} \
        --profile {PROFILE} \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --grad-accum-steps {GRAD_ACCUM_STEPS} \
        --precision {PRECISION} \
        --num-workers {NUM_WORKERS} \
        --checkpoint-dir {CKPT_BASE} \
        --output-report-dir {REPORT_BASE}
    print("✅ Hoàn tất Full ACD-MMKTv3 5-Fold trên Eedi!")

elif EXECUTION_MODE == "SINGLE_FOLD":
    print(f"🎯 Huấn luyện đơn lẻ Fold {TARGET_FOLD} cho Full ACD-MMKTv3 (Eedi)...")
    !python -u scripts/train_acd_mmkt_v3.py \
        --dataset {DATASET} \
        --test-split {TEST_SPLIT} \
        --post-diag-mode {POST_DIAG_MODE} \
        --val-fold {TARGET_FOLD} \
        --device {DEVICE_STR} \
        --profile {PROFILE} \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --grad-accum-steps {GRAD_ACCUM_STEPS} \
        --precision {PRECISION} \
        --num-workers {NUM_WORKERS} \
        --save-dir {CKPT_BASE}/acd_mmkt_v3_full_5fold/acd_mmkt_v3_fold{TARGET_FOLD}

elif EXECUTION_MODE == "QUICK_TEST":
    print("⚡ Chạy Quick Smoke Test (1 epoch trên Fold 0 Eedi)...")
    !python -u scripts/train_acd_mmkt_v3.py \
        --dataset {DATASET} \
        --post-diag-mode {POST_DIAG_MODE} \
        --val-fold 0 \
        --device {DEVICE_STR} \
        --profile {PROFILE} \
        --num-epochs 1 \
        --batch-size {BATCH_SIZE} \
        --grad-accum-steps 1 \
        --precision {PRECISION} \
        --num-workers 0 \
        --save-dir {CKPT_BASE}/quick_test_fold0
else:
    print(f"ℹ️ Bỏ qua huấn luyện Full Model theo EXECUTION_MODE={EXECUTION_MODE}.")

In [ ]:
# ==============================================================================
# [8] Modality Ablation 3: w/o Visual Stream (--ablation 3) — 5-Fold Dual GPU T4
# ==============================================================================
# Tắt hoàn toàn nhánh ảnh đề bài I_q; chỉ giữ C_tau, T_q, T_s và post_diag_vec.
# ==============================================================================
ABLATION_MODES = {"FULL_AND_MODALITY_ABLATIONS", "MODALITY_ABLATIONS_ONLY", "ALL_ABLATIONS_FULL_SUITE"}

if EXECUTION_MODE in ABLATION_MODES:
    eval_flag = "--eval-only" if (not FORCE_RETRAIN and all_5folds_exist("acd_mmkt_v3_ablation3_wo_visual_5fold")) else ""
    if eval_flag:
        print("📦 Đã có đủ 5 checkpoints Ablation 3 -> chạy --eval-only.")
    else:
        print("🚀 Bắt đầu huấn luyện Ablation 3 (w/o Visual Stream) 5-Fold trên Eedi...")

    !python -u scripts/run_5fold_acd_mmkt_v3.py \
        --dataset {DATASET} \
        --ablation 3 \
        --test-split {TEST_SPLIT} \
        --post-diag-mode {POST_DIAG_MODE} \
        {ACCEL_FLAG} \
        {eval_flag} \
        --profile {PROFILE} \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --grad-accum-steps {GRAD_ACCUM_STEPS} \
        --precision {PRECISION} \
        --num-workers {NUM_WORKERS} \
        --checkpoint-dir {CKPT_BASE} \
        --output-report-dir {REPORT_BASE}
    print("✅ Hoàn tất Ablation 3 (w/o Visual) trên Eedi!")
else:
    print(f"ℹ️ Bỏ qua Ablation 3 theo EXECUTION_MODE={EXECUTION_MODE}.")

In [ ]:
# ==============================================================================
# [9] Modality Ablation 2: w/o Solution Stream (--ablation 2) — 5-Fold Dual GPU T4
# ==============================================================================
# Tắt hoàn toàn nhánh lời giải tổng hợp VLM (T_s); chỉ giữ C_tau, T_q, I_q và post_diag_vec.
# ==============================================================================
if EXECUTION_MODE in ABLATION_MODES:
    eval_flag = "--eval-only" if (not FORCE_RETRAIN and all_5folds_exist("acd_mmkt_v3_ablation2_wo_solution_5fold")) else ""
    if eval_flag:
        print("📦 Đã có đủ 5 checkpoints Ablation 2 -> chạy --eval-only.")
    else:
        print("🚀 Bắt đầu huấn luyện Ablation 2 (w/o Solution Stream) 5-Fold trên Eedi...")

    !python -u scripts/run_5fold_acd_mmkt_v3.py \
        --dataset {DATASET} \
        --ablation 2 \
        --test-split {TEST_SPLIT} \
        --post-diag-mode {POST_DIAG_MODE} \
        {ACCEL_FLAG} \
        {eval_flag} \
        --profile {PROFILE} \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --grad-accum-steps {GRAD_ACCUM_STEPS} \
        --precision {PRECISION} \
        --num-workers {NUM_WORKERS} \
        --checkpoint-dir {CKPT_BASE} \
        --output-report-dir {REPORT_BASE}
    print("✅ Hoàn tất Ablation 2 (w/o Solution) trên Eedi!")
else:
    print(f"ℹ️ Bỏ qua Ablation 2 theo EXECUTION_MODE={EXECUTION_MODE}.")

In [ ]:
# ==============================================================================
# [10] Modality Ablation 1: Pure KC-Interaction / w/o All Multimodal (--ablation 1)
# ==============================================================================
# Tắt toàn bộ đa phương thức đề bài và lời giải; chỉ giữ C_tau, siêu đồ thị G và post_diag_vec.
# ==============================================================================
if EXECUTION_MODE in ABLATION_MODES:
    eval_flag = "--eval-only" if (not FORCE_RETRAIN and all_5folds_exist("acd_mmkt_v3_ablation1_wo_all_multimodal_5fold")) else ""
    if eval_flag:
        print("📦 Đã có đủ 5 checkpoints Ablation 1 -> chạy --eval-only.")
    else:
        print("🚀 Bắt đầu huấn luyện Ablation 1 (Pure KC-Interaction) 5-Fold trên Eedi...")

    !python -u scripts/run_5fold_acd_mmkt_v3.py \
        --dataset {DATASET} \
        --ablation 1 \
        --test-split {TEST_SPLIT} \
        --post-diag-mode {POST_DIAG_MODE} \
        {ACCEL_FLAG} \
        {eval_flag} \
        --profile {PROFILE} \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --grad-accum-steps {GRAD_ACCUM_STEPS} \
        --precision {PRECISION} \
        --num-workers {NUM_WORKERS} \
        --checkpoint-dir {CKPT_BASE} \
        --output-report-dir {REPORT_BASE}
    print("✅ Hoàn tất Ablation 1 (Pure KC) trên Eedi!")
else:
    print(f"ℹ️ Bỏ qua Ablation 1 theo EXECUTION_MODE={EXECUTION_MODE}.")

In [ ]:
# ==============================================================================
# [11] (Tùy Chọn) Architecture Ablations 4, 5, 6 trên Eedi (--ablation 4, 5, 6)
# ==============================================================================
# Chỉ chạy khi EXECUTION_MODE == "ALL_ABLATIONS_FULL_SUITE":
# - Ablation 4: Unanchored Concept Difficulty
# - Ablation 5: Polluted Experience Self-Attention
# - Ablation 6: w/o Monotonic 2PL-IRT Decoder
# ==============================================================================
if EXECUTION_MODE == "ALL_ABLATIONS_FULL_SUITE":
    arch_specs = [
        (4, "acd_mmkt_v3_ablation4_unanchored_5fold", "Ablation 4 (Unanchored Concept)"),
        (5, "acd_mmkt_v3_ablation5_polluted_exp_5fold", "Ablation 5 (Polluted Experience)"),
        (6, "acd_mmkt_v3_ablation6_wo_2pl_irt_5fold", "Ablation 6 (w/o 2PL-IRT)"),
    ]
    for ab_id, exp_dir, title in arch_specs:
        eval_flag = "--eval-only" if (not FORCE_RETRAIN and all_5folds_exist(exp_dir)) else ""
        print(f"🚀 Đang xử lý {title} trên Eedi (eval_only={bool(eval_flag)})...")
        !python -u scripts/run_5fold_acd_mmkt_v3.py \
            --dataset {DATASET} \
            --ablation {ab_id} \
            --test-split {TEST_SPLIT} \
            --post-diag-mode {POST_DIAG_MODE} \
            {ACCEL_FLAG} \
            {eval_flag} \
            --profile {PROFILE} \
            --num-epochs {NUM_EPOCHS} \
            --batch-size {BATCH_SIZE} \
            --lr {LR} \
            --grad-accum-steps {GRAD_ACCUM_STEPS} \
            --precision {PRECISION} \
            --num-workers {NUM_WORKERS} \
            --checkpoint-dir {CKPT_BASE} \
            --output-report-dir {REPORT_BASE}
        print(f"✅ Hoàn tất {title}!")
else:
    print(f"ℹ️ Bỏ qua Architecture Ablations 4/5/6 (chỉ chạy khi EXECUTION_MODE == 'ALL_ABLATIONS_FULL_SUITE').")

In [ ]:
# ==============================================================================
# [12] Tổng Hợp Master Modality Ablation Study & Kiểm Định Thống Kê 3 Tầng (Eedi)
# ==============================================================================
# Chạy run_acd_v3_modality_ablation_study.py ở chế độ --eval-only trên tập Test chính,
# đồng thời tự động đánh giá bổ sung trên Public và Private nếu EVAL_PUBLIC_PRIVATE=True.
# ==============================================================================
import os

available_mod_ab = []
for ab_id, folder in [
    ("3", "acd_mmkt_v3_ablation3_wo_visual_5fold"),
    ("2", "acd_mmkt_v3_ablation2_wo_solution_5fold"),
    ("1", "acd_mmkt_v3_ablation1_wo_all_multimodal_5fold"),
]:
    if all_5folds_exist(folder):
        available_mod_ab.append(ab_id)

has_full = all_5folds_exist("acd_mmkt_v3_full_5fold")

if has_full and len(available_mod_ab) > 0:
    ab_str = " ".join(available_mod_ab)
    splits_to_eval = [TEST_SPLIT]
    if EVAL_PUBLIC_PRIVATE and TEST_SPLIT == "unified":
        splits_to_eval.extend(["public", "private"])

    for sp in splits_to_eval:
        print(f"\n📊 Đang tổng hợp Master Modality Ablation Study cho tập Eedi '{sp}' (ablations: {available_mod_ab})...")
        !python -u scripts/run_acd_v3_modality_ablation_study.py \
            --dataset {DATASET} \
            --eval-only \
            --ablations {ab_str} \
            --test-split {sp} \
            --post-diag-mode {POST_DIAG_MODE} \
            --device {DEVICE_STR} \
            --test-batch-size {TEST_BATCH_SIZE} \
            --num-workers {NUM_WORKERS} \
            --checkpoint-dir {CKPT_BASE} \
            --output-report-dir {REPORT_BASE}
    print("✅ Hoàn tất tổng hợp Master Modality Ablation Study (Unified / Public / Private)!")
elif has_full and EVAL_PUBLIC_PRIVATE and TEST_SPLIT == "unified":
    # Nếu chỉ train Full Model (FULL_ONLY), vẫn tự động trích thêm điểm cho Public & Private
    for sp in ["public", "private"]:
        print(f"\n📊 Đang đánh giá bổ sung Full ACD-MMKTv3 trên tập Eedi '{sp}' (--eval-only)...")
        !python -u scripts/run_5fold_acd_mmkt_v3.py \
            --dataset {DATASET} \
            --eval-only \
            --test-split {sp} \
            --post-diag-mode {POST_DIAG_MODE} \
            --device {DEVICE_STR} \
            --profile {PROFILE} \
            --num-workers {NUM_WORKERS} \
            --checkpoint-dir {CKPT_BASE} \
            --output-report-dir {REPORT_BASE}
else:
    print("ℹ️ Chưa đủ checkpoints Full + Modality Ablations (3/2/1) để chạy tổng hợp đối chiếu.")

In [ ]:
# ==============================================================================
# [13] Đánh Giá Chi Tiết Eedi Strata, Task 3 Expert Pairs & Giải Thích 4 Tầng
# ==============================================================================
import os

eval_targets = [
    ("full", os.path.join(CKPT_BASE, f"acd_mmkt_v3_full_5fold/acd_mmkt_v3_fold{TARGET_FOLD}/best_model.pt")),
    ("ab3_wovis", os.path.join(CKPT_BASE, f"acd_mmkt_v3_ablation3_wo_visual_5fold/acd_mmkt_v3_fold{TARGET_FOLD}/best_model.pt")),
    ("ab2_wosol", os.path.join(CKPT_BASE, f"acd_mmkt_v3_ablation2_wo_solution_5fold/acd_mmkt_v3_fold{TARGET_FOLD}/best_model.pt")),
    ("ab1_purekc", os.path.join(CKPT_BASE, f"acd_mmkt_v3_ablation1_wo_all_multimodal_5fold/acd_mmkt_v3_fold{TARGET_FOLD}/best_model.pt")),
    ("quick_test", os.path.join(CKPT_BASE, "quick_test_fold0/best_model.pt")),
]

evaluated_any = False
for tag, ckpt_path in eval_targets:
    if tag == "quick_test" and evaluated_any:
        continue
    if os.path.exists(ckpt_path):
        evaluated_any = True
        rep_json = os.path.join(REPORT_BASE, f"eval_eedi_{tag}.json")
        exp_md = os.path.join(REPORT_BASE, f"explanation_sample_eedi_{tag}.md")
        use_fold = 0 if tag == "quick_test" else TARGET_FOLD
        print(f"🔍 Đánh giá chi tiết & Explainability 4 tầng cho [{tag}] từ {ckpt_path}...")
        !python -u scripts/evaluate_acd_mmkt_v3.py \
            --dataset {DATASET} \
            --checkpoint {ckpt_path} \
            --val-fold {use_fold} \
            --test-split {TEST_SPLIT} \
            --device {DEVICE_STR} \
            --batch-size {TEST_BATCH_SIZE} \
            --test-stride {TEST_STRIDE} \
            --num-workers {NUM_WORKERS} \
            --output-report {rep_json} \
            --explain-output-md {exp_md}

if not evaluated_any:
    print("⚠️ Chưa tìm thấy checkpoint nào để đánh giá chi tiết.")

In [ ]:
# ==============================================================================
# [14] Hiển Thị Bảng Tổng Hợp Kết Quả Thực Nghiệm Eedi (Unified / Public / Private)
# ==============================================================================
import json
import os
import pandas as pd
from IPython.display import display

split_tag = "" if TEST_SPLIT == "unified" else f"_{TEST_SPLIT}"
master_json = os.path.join(REPORT_BASE, f"acd_mmkt_v3_modality_ablation_study_comprehensive{split_tag}.json")
full_rep_json = os.path.join(REPORT_BASE, f"acd_mmkt_v3_5fold_report{split_tag}.json")
eval_full_json = os.path.join(REPORT_BASE, "eval_eedi_full.json")

# 1. Bảng tổng hợp 5-Fold CV nếu có Master Ablation JSON
if os.path.exists(master_json):
    with open(master_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    fm = data.get("full_model", {})
    ab = data.get("ablations", {})

    rows_global = []
    if "mean_metrics" in fm and "ensemble_metrics" in fm:
        em = fm["ensemble_metrics"]
        mm = fm["mean_metrics"]
        rows_global.append({
            "Mô hình / Biến thể": "Full ACD-MMKTv3 (Eedi)",
            "Phương thức": "C_tau, T_q, I_q, T_s",
            "Tham số": "762,376",
            "5-Fold Mean AUC": f"{mm['auc']:.4f} ± {mm['auc_std']:.4f}",
            "Ensemble AUC": f"{em['auc']:.4f}",
            "Ensemble ACC": f"{em['acc']:.4f}",
            "RMSE": f"{em['rmse']:.4f}",
            "Brier": f"{em['brier']:.4f}",
            "ECE": f"{em['ece']:.4f}",
            "Macro-AUC": f"{em.get('student_macro_auc', 0.0):.4f}",
        })
    for k, mod_desc in [("3", "C_tau, T_q, T_s"), ("2", "C_tau, T_q, I_q"), ("1", "Chỉ C_tau (Siêu đồ thị G)")]:
        if k in ab:
            item = ab[k]
            mm, em = item["mean_metrics"], item["ensemble_metrics"]
            rows_global.append({
                "Mô hình / Biến thể": item["display_name"],
                "Phương thức": mod_desc,
                "Tham số": f"{item['total_params']:,}",
                "5-Fold Mean AUC": f"{mm['auc']:.4f} ± {mm['auc_std']:.4f}",
                "Ensemble AUC": f"{em['auc']:.4f}",
                "Ensemble ACC": f"{em['acc']:.4f}",
                "RMSE": f"{em['rmse']:.4f}",
                "Brier": f"{em['brier']:.4f}",
                "ECE": f"{em['ece']:.4f}",
                "Macro-AUC": f"{em.get('student_macro_auc', 0.0):.4f}",
            })
    print(f"🏆 1. HIỆU NĂNG 5-FOLD TRÊN EEDI ({TEST_SPLIT.upper()}):")
    display(pd.DataFrame(rows_global))

    # 2. Bảng Phân tầng Eedi (Thị giác, Hình thái Slide, Miền Toán học, Misconception)
    full_strat = fm.get("stratified_metrics", {})
    ab3_strat = ab.get("3", {}).get("stratified_metrics", {})
    ab1_strat = ab.get("1", {}).get("stratified_metrics", {})
    strat_keys = [
        ("overall", "Toàn cục (Overall)"),
        ("q_stem_image", "Có ảnh đề bài (m_img^q = 1)"),
        ("q_text_only", "Thuần văn bản (m_img^q = 0)"),
        ("vis_scaffold_aid", "Visual Scaffold (Delta mu_vis < 0)"),
        ("vis_cognitive_overload", "Visual Overload (Delta mu_vis > 0)"),
        ("layout_horizontal", "Layout Slide: Horizontal"),
        ("layout_grid", "Layout Slide: Grid 2x2"),
        ("layout_vertical", "Layout Slide: Vertical"),
        ("domain_number", "Miền: Number"),
        ("domain_algebra", "Miền: Algebra"),
        ("domain_geometry", "Miền: Geometry & Measure"),
        ("after_dom_misconception", "Sau lỗi Misconception chủ đạo"),
        ("after_overconfident_error", "Sau lỗi Tự tin quá mức"),
        ("freq_rare_le10", "Câu hỏi hiếm / Cold-Start (N_q <= 10)"),
    ]
    s_rows = []
    for sk, label in strat_keys:
        if sk in full_strat and full_strat[sk].get("num_samples", 0) > 0:
            row = {
                "Lát cắt Sư phạm Eedi": label,
                "Số mẫu": f"{full_strat[sk]['num_samples']:,}",
                "Full AUC": f"{full_strat[sk]['auc']:.4f}",
            }
            if sk in ab3_strat:
                row["Abl.3 (w/o Vis)"] = f"{ab3_strat[sk]['auc']:.4f} ({ab3_strat[sk]['auc'] - full_strat[sk]['auc']:+.4f})"
            if sk in ab1_strat:
                row["Abl.1 (Pure KC)"] = f"{ab1_strat[sk]['auc']:.4f} ({ab1_strat[sk]['auc'] - full_strat[sk]['auc']:+.4f})"
            s_rows.append(row)
    print("\n📊 2. PHÂN TẦNG ĐA CHIỀU ĐẶC THÙ EEDI:")
    display(pd.DataFrame(s_rows))

    # 3. Kiểm định thống kê 3 tầng
    delong_all = data.get("delong_tests_overall", {})
    boot_pw = data.get("cluster_bootstrap_95ci", {}).get("pairwise", {})
    st_rows = []
    for label, m_a, m_b, ab_key in [
        ("Full vs Ablation 3 (w/o Visual)", "Full ACD-MMKTv3", "Ablation 3 (w/o Visual)", "3"),
        ("Full vs Ablation 2 (w/o Solution)", "Full ACD-MMKTv3", "Ablation 2 (w/o Solution)", "2"),
        ("Full vs Ablation 1 (Pure KC)", "Full ACD-MMKTv3", "Ablation 1 (Pure KC)", "1"),
    ]:
        st = ab.get(ab_key, {}).get("statistical_tests", {}).get("Full ACD-MMKTv3", {})
        del_res = delong_all.get(f"{m_a}__vs__{m_b}", delong_all.get(f"{m_b}__vs__{m_a}", {}))
        boot_res = boot_pw.get(f"{m_a}__vs__{m_b}", boot_pw.get(f"{m_b}__vs__{m_a}", {}))
        if st or del_res:
            st_rows.append({
                "Cặp đối chiếu": label,
                "Paired t p-val": f"{st.get('paired_t_pval', 1.0):.2e}" if st else "N/A",
                "Wilcoxon p-val": f"{st.get('wilcoxon_pval', 1.0):.2e}" if st else "N/A",
                "DeLong p-adj": f"{del_res.get('p_value_holm', del_res.get('p_value', 1.0)):.2e}" if del_res else "N/A",
                "95% Bootstrap CI [ΔAUC]": f"[{boot_res.get('delta_auc_ci_low', 0.0):+.4f}, {boot_res.get('delta_auc_ci_high', 0.0):+.4f}]" if boot_res else "N/A",
            })
    if st_rows:
        print("\n📐 3. KIỂM ĐỊNH Ý NGHĨA THỐNG KÊ 3 TẦNG:")
        display(pd.DataFrame(st_rows))

elif os.path.exists(full_rep_json):
    with open(full_rep_json, "r", encoding="utf-8") as f:
        rep = json.load(f)
    print(f"🏆 Full ACD-MMKTv3 Eedi ({TEST_SPLIT}): Mean AUC = {rep['mean_metrics']['auc']:.4f} ± {rep['mean_metrics']['auc_std']:.4f} | Ensemble AUC = {rep['ensemble_5fold_metrics']['auc']:.4f}")

# 4. Tổng hợp so sánh nhanh Unified vs Public vs Private (nếu đã đánh giá)
split_rows = []
for sp_name, sp_suf in [("Unified (124,960)", ""), ("Public (74,276)", "_public"), ("Private (50,684)", "_private")]:
    p_master = os.path.join(REPORT_BASE, f"acd_mmkt_v3_modality_ablation_study_comprehensive{sp_suf}.json")
    p_full = os.path.join(REPORT_BASE, f"acd_mmkt_v3_5fold_report{sp_suf}.json")
    if os.path.exists(p_master):
        with open(p_master, "r", encoding="utf-8") as f:
            dm = json.load(f)
        row = {"Phân vùng Test Eedi": sp_name, "Full Ens. AUC": f"{dm['full_model']['ensemble_metrics']['auc']:.4f}"}
        for k, col in [("3", "Abl.3 (w/o Vis)"), ("2", "Abl.2 (w/o Sol)"), ("1", "Abl.1 (Pure KC)")]:
            if k in dm.get("ablations", {}):
                row[col] = f"{dm['ablations'][k]['ensemble_metrics']['auc']:.4f}"
        split_rows.append(row)
    elif os.path.exists(p_full):
        with open(p_full, "r", encoding="utf-8") as f:
            df = json.load(f)
        split_rows.append({"Phân vùng Test Eedi": sp_name, "Full Ens. AUC": f"{df['ensemble_5fold_metrics']['auc']:.4f}"})
if split_rows:
    print("\n🔀 4. ĐỐI CHIẾU HIỆU NĂNG TRÊN 3 TẬP TEST EEDI (UNIFIED / PUBLIC / PRIVATE):")
    display(pd.DataFrame(split_rows))

# 5. Kết quả kiểm định Chuyên gia Task 3 & Miền Toán học
if os.path.exists(eval_full_json):
    with open(eval_full_json, "r", encoding="utf-8") as f:
        ev = json.load(f)
    t3 = ev.get("task3_expert_validation", {})
    if t3:
        print("\n🧑‍🏫 5. ĐỐI CHIẾU CHUYÊN GIA EEDI TASK 3 (50 CẶP SO SÁNH CHẤT LƯỢNG):")
        display(pd.DataFrame([t3]))

In [ ]:
# ==============================================================================
# [15] Trực Quan Hóa Biểu Đồ Nghiên Cứu Eedi từ Dữ Liệu Thực Tế (300 DPI)
# ==============================================================================
import json
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(FIG_BASE, exist_ok=True)
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({"font.size": 10.5, "axes.labelsize": 11, "axes.titlesize": 12})

split_tag = "" if TEST_SPLIT == "unified" else f"_{TEST_SPLIT}"
master_json = os.path.join(REPORT_BASE, f"acd_mmkt_v3_modality_ablation_study_comprehensive{split_tag}.json")
eval_full_json = os.path.join(REPORT_BASE, "eval_eedi_full.json")

if os.path.exists(master_json) or os.path.exists(eval_full_json):
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 11.5))

    m_data = json.load(open(master_json, "r", encoding="utf-8")) if os.path.exists(master_json) else {}
    ev_data = json.load(open(eval_full_json, "r", encoding="utf-8")) if os.path.exists(eval_full_json) else {}

    # (a) So sánh Ensemble AUC trên Eedi giữa Full & các biến thể Ablation có sẵn
    labels_a, aucs_a, colors_a = [], [], []
    if "full_model" in m_data:
        labels_a.append("Full ACD-MMKTv3")
        aucs_a.append(m_data["full_model"]["ensemble_metrics"]["auc"])
        colors_a.append("#d62728")
    elif "global_metrics" in ev_data:
        labels_a.append("Full (Single Fold)")
        aucs_a.append(ev_data["global_metrics"]["auc"])
        colors_a.append("#d62728")
    for k, lbl, col in [
        ("2", "Abl.2 (w/o Sol)", "#2ca02c"),
        ("3", "Abl.3 (w/o Vis)", "#ff7f0e"),
        ("1", "Abl.1 (Pure KC)", "#1f77b4"),
    ]:
        if k in m_data.get("ablations", {}):
            labels_a.append(lbl)
            aucs_a.append(m_data["ablations"][k]["ensemble_metrics"]["auc"])
            colors_a.append(col)
    if labels_a:
        bars = ax1.bar(labels_a, aucs_a, color=colors_a, width=0.5, edgecolor="black", linewidth=0.8)
        y_min = max(0.5, min(aucs_a) - 0.03)
        y_max = min(1.0, max(aucs_a) + 0.02)
        ax1.set_ylim(y_min, y_max)
        for b, v in zip(bars, aucs_a):
            ax1.text(b.get_x() + b.get_width() / 2.0, v + 0.0015, f"{v:.4f}", ha="center", va="bottom", fontweight="bold")
    ax1.set_title("(a) Eedi Test ROC-AUC Comparison (Real Outputs)", fontweight="bold")
    ax1.set_ylabel("ROC-AUC")
    ax1.grid(axis="y", linestyle="--", alpha=0.6)

    # (b) Phân tầng Thị giác & Hình thái Slide (Full vs Ablation 3)
    full_strat = m_data.get("full_model", {}).get("stratified_metrics", {})
    ab3_strat = m_data.get("ablations", {}).get("3", {}).get("stratified_metrics", {})
    vis_keys = [
        ("q_stem_image", "Stem Image"),
        ("vis_scaffold_aid", "Scaffold (<0)"),
        ("vis_cognitive_overload", "Overload (>0)"),
        ("layout_horizontal", "Horizontal"),
        ("layout_grid", "Grid 2x2"),
        ("layout_vertical", "Vertical"),
    ]
    vk_avail = [(k, lbl) for k, lbl in vis_keys if k in full_strat and full_strat[k].get("num_samples", 0) > 0]
    if vk_avail:
        x = np.arange(len(vk_avail))
        w = 0.36
        v_full = [full_strat[k]["auc"] for k, _ in vk_avail]
        ax2.bar(x - (w / 2 if ab3_strat else 0), v_full, w, label="Full ACD-MMKTv3", color="#d62728", edgecolor="black", linewidth=0.8)
        if ab3_strat:
            v_ab3 = [ab3_strat.get(k, {}).get("auc", 0.0) for k, _ in vk_avail]
            ax2.bar(x + w / 2, v_ab3, w, label="Ablation 3 (w/o Visual)", color="#ff7f0e", edgecolor="black", linewidth=0.8)
        ax2.set_xticks(x)
        ax2.set_xticklabels([lbl for _, lbl in vk_avail], rotation=15, ha="right")
        ax2.set_ylim(max(0.5, min(v_full) - 0.04), min(1.0, max(v_full) + 0.03))
        ax2.legend(loc="lower right", frameon=True)
    ax2.set_title("(b) Visual Slices & Slide Layouts on Eedi", fontweight="bold")
    ax2.set_ylabel("Ensemble ROC-AUC")
    ax2.grid(axis="y", linestyle="--", alpha=0.6)

    # (c) Miền Toán học, Misconception & Cold-Start (Full vs Ablation 1 Pure KC)
    ab1_strat = m_data.get("ablations", {}).get("1", {}).get("stratified_metrics", {})
    eedi_keys = [
        ("domain_number", "Number"),
        ("domain_algebra", "Algebra"),
        ("domain_geometry", "Geometry"),
        ("after_dom_misconception", "After Miscon."),
        ("after_overconfident_error", "Overconf. Err"),
        ("freq_rare_le10", "Cold-Start<=10"),
    ]
    ek_avail = [(k, lbl) for k, lbl in eedi_keys if k in full_strat and full_strat[k].get("num_samples", 0) > 0]
    if ek_avail:
        x3 = np.arange(len(ek_avail))
        w = 0.36
        vf3 = [full_strat[k]["auc"] for k, _ in ek_avail]
        ax3.bar(x3 - (w / 2 if ab1_strat else 0), vf3, w, label="Full ACD-MMKTv3", color="#d62728", edgecolor="black", linewidth=0.8)
        if ab1_strat:
            va1 = [ab1_strat.get(k, {}).get("auc", 0.0) for k, _ in ek_avail]
            ax3.bar(x3 + w / 2, va1, w, label="Ablation 1 (Pure KC)", color="#1f77b4", edgecolor="black", linewidth=0.8)
        ax3.set_xticks(x3)
        ax3.set_xticklabels([lbl for _, lbl in ek_avail], rotation=15, ha="right")
        ax3.set_ylim(max(0.5, min(vf3) - 0.05), min(1.0, max(vf3) + 0.03))
        ax3.legend(loc="lower right", frameon=True)
    ax3.set_title("(c) Domains, Misconceptions & Cold-Start Robustness", fontweight="bold")
    ax3.set_ylabel("Ensemble ROC-AUC")
    ax3.grid(axis="y", linestyle="--", alpha=0.6)

    # (d) Hiệu chuẩn Tâm trắc học THỰC TẾ trên 948 câu hỏi Eedi (tilde_mu vs D_q thực nghiệm)
    props_path = os.path.join(CKPT_BASE, "acd_mmkt_v3_full_5fold/full_item_props_5fold.pt")
    aux_path = "data/processed/Eedi/acd_v3_aux_metadata.pt"
    if os.path.exists(props_path) and os.path.exists(aux_path):
        cached_p = torch.load(props_path, map_location="cpu", weights_only=False)
        aux_d = torch.load(aux_path, map_location="cpu", weights_only=False)
        mean_tilde_mu = np.mean([fp["tilde_mu"] for fp in cached_p["fold_props"]], axis=0)
        emp_err = aux_d["D_q_folds"][0].numpy()
        valid_q = aux_d["N_q_folds"][0].numpy() >= 10
        x_d, y_mu = emp_err[valid_q], mean_tilde_mu[valid_q]
        r_val = float(np.corrcoef(x_d, y_mu)[0, 1])
        ax4.scatter(x_d, y_mu, alpha=0.55, color="#1f77b4", s=24)
        m_sl, b_ic = np.polyfit(x_d, y_mu, 1)
        ax4.plot(x_d, m_sl * x_d + b_ic, color="#d62728", linewidth=2.0, label=f"Pearson r = {r_val:+.4f} (N={len(x_d)})")
        ax4.legend(loc="upper left", frameon=True)
        ax4.set_xlabel("Empirical Training Error Rate (D_q)")
        ax4.set_ylabel("Analytical Difficulty Logit (tilde_mu_q)")
    else:
        dom_stats = ev_data.get("domain_gate_intensity", {})
        if dom_stats:
            d_names = list(dom_stats.keys())
            d_means = [dom_stats[k]["mean_lambda"] for k in d_names]
            ax4.barh(d_names, d_means, color="#2ca02c", edgecolor="black", linewidth=0.8)
            ax4.set_xlabel("Mean Visual Gate Intensity (bar_lambda_q)")
    ax4.set_title("(d) Eedi Psychometric / Visual Gate Analysis", fontweight="bold")
    ax4.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    fig_path = os.path.join(FIG_BASE, f"acd_mmkt_v3_eedi_analysis{split_tag}.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✅ Đã lưu biểu đồ chất lượng xuất bản tại: {fig_path}")
else:
    print("ℹ️ Chưa có file JSON báo cáo kết quả Eedi để vẽ biểu đồ thực tế.")

In [ ]:
# ==============================================================================
# [16] Đóng Gói Toàn Bộ Checkpoints, Reports, Logs và Figures Eedi Thành Tệp ZIP
# ==============================================================================
import os

zip_dest = "/kaggle/working/acd_mmkt_v3_eedi_results.zip"
paths_to_zip = [p for p in [CKPT_BASE, REPORT_BASE, LOG_BASE] if os.path.exists(p)]
if paths_to_zip:
    joined = " ".join(paths_to_zip)
    print(f"📦 Đang nén toàn bộ kết quả Eedi vào: {zip_dest}...")
    !zip -q -r {zip_dest} {joined}
    if os.path.exists(zip_dest):
        zip_size_mb = os.path.getsize(zip_dest) / (1024**2)
        print(f"🎉 Đã nén thành công: {zip_dest} ({zip_size_mb:.2f} MB)")
        print("👉 Bạn có thể tải trực tiếp file ZIP này từ tab Output của Kaggle!")
    else:
        print("⚠️ Không tạo được file ZIP. Vui lòng kiểm tra dung lượng đĩa còn lại.")
else:
    print("⚠️ Chưa tìm thấy thư mục output của Eedi để đóng gói.")